In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder\
    .appName("recomendacao")\
    .master("spark://spark-master:7077")\
    .getOrCreate()

spark.sparkContext.setLogLevel("OFF")

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/11 19:30:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
raw_df_alura = spark.read.csv("s3a://bronze/recomendacao/alura", sep=";", header=True, inferSchema=True)
raw_df_kagle = spark.read.csv("s3a://bronze/recomendacao/kaggle-spotify", sep=",", header=True, inferSchema=True)

In [12]:
raw_df_kagle.count()


114000

In [5]:
from pyspark.sql.functions import col, concat_ws
from pyspark.sql.types import DoubleType

df = (
    raw_df_kagle
    .withColumnRenamed("track_id", "id")
    .withColumnRenamed("track_name", "name")
    .withColumn("artists_song", concat_ws(" - ", "artists", "name"))
)
cols = [
    "valence","acousticness", "artists", "danceability", "duration_ms",
    "energy", "explicit", "id", "instrumentalness", "key", "liveness", "loudness",
    "mode", "name", "popularity", "speechiness", "tempo", "artists_song"
]
numeric_str_cols = [
    "valence", "acousticness", "danceability", "duration_ms",
    "energy", "liveness", "loudness", "speechiness", "instrumentalness", "key", "tempo",
    "mode", "popularity"
]
    
for c in numeric_str_cols:
    df = df.withColumn(c, col(c).cast(DoubleType()))

df = df.select(cols)

In [6]:
df.printSchema()
raw_df_alura.printSchema()

root
 |-- valence: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- artists: string (nullable = true)
 |-- danceability: double (nullable = true)
 |-- duration_ms: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- explicit: string (nullable = true)
 |-- id: string (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- key: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: double (nullable = true)
 |-- name: string (nullable = true)
 |-- popularity: double (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- artists_song: string (nullable = false)

root
 |-- valence: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- artists: string (nullable = true)
 |-- danceability: double (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- energy: double

In [7]:
man = spark.read.parquet("s3a://silver/recomendacao/kaggle_spotify")

In [8]:
man.show(5)


+--------------------+-------+------------+--------------------+------------+-----------+------+--------+----------------+----+--------+--------+----+--------------------+----------+-----------+-------+--------------------+----+
|                  id|valence|acousticness|             artists|danceability|duration_ms|energy|explicit|instrumentalness| key|liveness|loudness|mode|                name|popularity|speechiness|  tempo|        artists_song|year|
+--------------------+-------+------------+--------------------+------------+-----------+------+--------+----------------+----+--------+--------+----+--------------------+----------+-----------+-------+--------------------+----+
|2UOtnKZNppcZ5dC8v...|  0.057|       0.276|        Nora En Pure|       0.537|   193403.0| 0.782|   False|           0.841|11.0|   0.132|   -7.91| 0.0|   Sign of the Times|      46.0|     0.0801|125.999|Nora En Pure - Si...|2021|
|3eCLAuvGkXm3ysWPu...|  0.663|     1.33E-4|          Aaron Carl|       0.797|   2862

In [9]:
man.printSchema()

root
 |-- id: string (nullable = true)
 |-- valence: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- artists: string (nullable = true)
 |-- danceability: double (nullable = true)
 |-- duration_ms: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- explicit: string (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- key: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: double (nullable = true)
 |-- name: string (nullable = true)
 |-- popularity: double (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- artists_song: string (nullable = true)
 |-- year: long (nullable = true)



In [15]:
df_cleaned = df.drop_duplicates(subset=["id"])
df_cleaned.count()

89741

In [16]:
raw_df_kagle.show(5)

+---+--------------------+--------------------+--------------------+--------------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+
|_c0|            track_id|             artists|          album_name|          track_name|popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|  tempo|time_signature|track_genre|
+---+--------------------+--------------------+--------------------+--------------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+
|  0|5SuOikwiRyPMVoIQD...|         Gen Hoshino|              Comedy|              Comedy|        73|     230666|   False|       0.676| 0.461|  1|  -6.746|   0|      0.143|      0.0322|         1.01E-6|   0.358|  0.715| 87.917|           4

In [17]:
df_cleaned2 = raw_df_kagle.drop_duplicates(subset=["track_id"])
df_cleaned2.count()

89741